# 🔒 Zero-Trust Supervised Verification (Kaggle Edition)

This notebook unpacks the ChakraModel zero-trust verification harness and runs it on the Kaggle datasets. By using Kaggle's dual-GPU environment, we can rapidly process massive datasets like HyperKvasir and SUN-SEG.

In [ ]:
!pip install -q ultralytics timm albumentations opencv-python-headless

In [ ]:
import os
import shutil
import subprocess

# Extract the zero-trust code bundle (assuming it's attached as a dataset or uploaded to working directory)
zip_path = None
for p in ['/kaggle/working/Kaggle_ZeroTrust_Code.zip', '/kaggle/input/Kaggle_ZeroTrust_Code.zip']:
    if os.path.exists(p):
        zip_path = p
        break
if not zip_path:
    # Search recursively in input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'Kaggle_ZeroTrust_Code.zip' in files:
            zip_path = os.path.join(root, 'Kaggle_ZeroTrust_Code.zip')
            break

if zip_path:
    print(f'Unpacking code from {zip_path}')
    !unzip -q -o "{zip_path}" -d /kaggle/working/
else:
    print('WARNING: Kaggle_ZeroTrust_Code.zip not found! Please attach it.')


In [ ]:
# Link model weights to /kaggle/working/weights so the script finds them
os.makedirs('/kaggle/working/weights', exist_ok=True)
weight_targets = {'chakra_transformer_best.pth', 'best.pt'}
found_weights = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in weight_targets and f not in found_weights:
            src = os.path.join(root, f)
            dst = os.path.join('/kaggle/working/weights', f)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
            found_weights.add(f)
            print(f'Linked {f} from {src}')

if len(found_weights) < 2:
    print('WARNING: Missing some weights! Ensure both chakra_transformer_best.pth and best.pt are uploaded.')


In [ ]:
import glob

# The 5 verified dataset folders uploaded so far (from Google Sheet)
target_datasets = [
    'LDPolypVideo_Polyp_Raw',
    'EndoScene_CVC300_Polyp_Raw',
    'PICCOLO_Dataset_Polyp_Raw',
    'PolypDB_Polyp_Raw',
    'HyperKvasir_Segmented_Raw'
]

print('================================================')
print('   RUNNING ZERO-TRUST SUPERVISED VERIFICATION')
print('================================================\n')

for dataset_name in target_datasets:
    # Find the dataset path in /kaggle/input
    # Because Kaggle lowercases dataset folder names, we do a case-insensitive search
    dataset_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if d.lower().replace('-', '_') == dataset_name.lower().replace('-', '_'):
                dataset_path = os.path.join(root, d)
                break
        if dataset_path: break

    if not dataset_path:
        print(f'\n[SKIPPED] Dataset {dataset_name} not found in /kaggle/input (Did you attach it?)')
        continue

    print(f'\n>> VERIFYING {dataset_name} at {dataset_path}')
    
    # The harness expects an absolute path and passes it via DATASET_ROOT
    env = os.environ.copy()
    env['DATASET_ROOT'] = dataset_path
    
    # Fix working directory for imports to resolve correctly
    cwd = '/kaggle/working'
    script_harness = '.agents/verification/supervised_verification_harness.py'
    script_eval = '.agents/verification/verify_eval.py'

    cmd = [
        'python', script_harness,
        '--dataset-root', dataset_path,
        '--script', script_eval
    ]

    try:
        # Execute the harness natively
        subprocess.run(cmd, env=env, cwd=cwd, check=True)
    except subprocess.CalledProcessError as e:
        print(f'\n[ERROR] Verification failed for {dataset_name}')

print('\n================================================')
print('VERIFICATION COMPLETE! Check /kaggle/working/verification_verdicts for Signed Cards.')
!ls -la /kaggle/working/verification_verdicts
